In [2]:
import pyart
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import cartopy.feature as cfeature
import xarray as xr
import os
import imageio.v2 as imageio
import pandas as pd
import time
import io
import cmweather
from datetime import datetime

In [51]:
# Function that tells plotting routine where to place aircraft relative to given radar scan 
def air_radar_time(cit_times, radar_times):
    air_radar_index = []
    cit_index = []
    for x in range(len(radar_times) - 1):
        for y in range(len(cit_times)):
            if datetime.strptime(cit_times[y][0:19], "%Y-%m-%dT%H:%M:%S").timestamp() >= datetime.strptime(radar_times[x].name[21:36], "%Y%m%d.%H%M%S").timestamp() and datetime.strptime(cit_times[y][0:19], "%Y-%m-%dT%H:%M:%S").timestamp() <= datetime.strptime(radar_times[x + 1].name[21:36], "%Y%m%d.%H%M%S").timestamp():
                cit_index.append(y)
        air_radar_indexs = air_radar_index.append(cit_index)
        cit_index = []
    return air_radar_index

In [60]:
# Initial reading in and parsing of flight data 
citation_nav = pd.read_csv('flight_data/2011-05-20_CITATION.dat', sep = ',', header = None)
cit_time = citation_nav.iloc[:, 1]
cit_lat = citation_nav.iloc[:, 2]
cit_lon = citation_nav.iloc[:, 3]

# List generation of possible radar files to plot 
with os.scandir('C-SAPR_May_20_2011/') as entries:
    radar_files = list(entries)

# Calling of previous function to get a list of valid times for aircraft plotting 
air_radar_list = air_radar_time(cit_time, radar_files)

# Define the gif writer
with imageio.get_writer('radar_images/May_20_2011_C-SAPR_aircraft3.gif', mode='I', loop = 0, duration = 500) as writer:
    for x in range(len(air_radar_list)):
        filename = radar_files[x].name
        data = pyart.io.read('C-SAPR_May_20_2011/' + filename)
        for y in air_radar_list[x]:
            
            # Reflectivity plotting 
            display = pyart.graph.RadarMapDisplay(data)
                                    
            projection = ccrs.PlateCarree()
                        
            fig = plt.figure(figsize = (11,9))
                                    
            # Reflectivity plot
            ax = fig.add_axes([0.06, 0.50, 0.40, 0.40], projection = projection)
            ax.add_feature(cfeature.STATES.with_scale('10m'), linewidth = 1.5, edgecolor = 'black')
                                    
            display.plot_ppi_map('corrected_reflectivity_horizontal', 
                                ax = ax,
                                sweep = 1, 
                                vmin = -30, 
                                vmax = 70,
                                cmap = 'ChaseSpectral') #Variable name

            # Plotting code for aircraft location along with a track of where the aircraft had all went throughout the scan interval 
            arcft_pos = (cit_lon[y], cit_lat[y])
            ax.annotate('\u2708', xy = arcft_pos, fontsize = 12, ha = 'center', va = 'center')
            plt.plot(cit_lon[min(air_radar_list[x]):y], cit_lat[min(air_radar_list[x]):y])
            
            plt.tight_layout()

            # Temporary saving of matplotlib file 
            buf = io.BytesIO()
            fig.savefig(buf, format="png", bbox_inches="tight")
            buf.seek(0)
            writer.append_data(imageio.imread(buf))
            buf.close()
            plt.close(fig)